# MODIS MCD15A2H (LAI/FPAR) — Dataset Facts

Structural facts only: shape, missingness, timing, value ranges, and
quality breakdown for the 5 downloaded MODIS pixels. No correlation or
modeling here -- see `02-correlation-analysis` for that.


In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import rasterio

# vvv EDIT THIS to match your folder vvv
MODIS_DIR = "modis_downloads"
# ^^^ EDIT THIS to match your folder ^^^

SCALE = {"Lai_500m": 0.1, "Fpar_500m": 0.01, "LaiStdDev_500m": 0.01, "FparStdDev_500m": 0.01}
FILL_VALUES = {249, 250, 251, 252, 253, 254, 255}


## 1. Load every layer

Reads each file at array cell `[1,1]` -- the confirmed real value in
each 2x2 GeoTIFF (the other 3 cells are the MODIS fill value).

In [ ]:
all_files = glob.glob(os.path.join(MODIS_DIR, "**", "*.tif"), recursive=True)
print(f"Total .tif files found: {len(all_files)}")

def load_layer(layer_name, apply_scale=True):
    pattern = re.compile(rf"MCD15A2H\.061_{re.escape(layer_name)}_(\d{{8}})T\d+_aid(\d+)")
    files = [p for p in all_files if layer_name in os.path.basename(p)]
    records = []
    for path in files:
        fname = os.path.basename(path)
        m = pattern.search(fname)
        if not m:
            continue
        date_str, pixel_num = m.groups()
        with rasterio.open(path) as src:
            arr = src.read(1)
            raw = arr[1, 1] if arr.shape == (2, 2) else np.nan
        is_fill = raw in FILL_VALUES
        value = np.nan if is_fill else raw * SCALE.get(layer_name, 1.0) if apply_scale else raw
        records.append({"date": pd.to_datetime(date_str, format="%Y%m%d"),
                         "pixel": f"pixel_{int(pixel_num)}", "raw": raw, "is_fill": is_fill, "value": value})
    return pd.DataFrame(records)

layers = {}
for layer in ["Lai_500m", "Fpar_500m", "FparLai_QC", "LaiStdDev_500m", "FparStdDev_500m"]:
    layers[layer] = load_layer(layer, apply_scale=(layer != "FparLai_QC"))
    print(f"{layer}: {len(layers[layer])} rows")


## 2. Shape and pixel list

In [ ]:
for name, df in layers.items():
    if len(df) == 0:
        continue
    print(f"{name}: {df['date'].nunique()} dates x {df['pixel'].nunique()} pixels "
          f"({len(df)} total rows)")
print(f"\nPixels found: {sorted(layers['Lai_500m']['pixel'].unique())}")


## 3. Date range

In [ ]:
for name, df in layers.items():
    if len(df) == 0:
        continue
    print(f"{name}: {df['date'].min().date()} to {df['date'].max().date()}")


## 4. Missing / fill values

In [ ]:
for name, df in layers.items():
    if len(df) == 0:
        continue
    n_fill = df["is_fill"].sum()
    print(f"{name}: {n_fill}/{len(df)} rows are fill/no-data ({100*n_fill/len(df):.1f}%)")


## 5. Timing between composite dates

Uses the correct **true per-year 8-day MODIS grid**, not a naive fixed
8-day-from-start assumption -- the two disagree slightly at year
boundaries.

In [ ]:
dates_sorted = sorted(layers["Lai_500m"]["date"].unique())
dates_sorted = pd.Series(pd.to_datetime(dates_sorted))
diffs = dates_sorted.diff().dt.days.dropna()
print("Gap distribution (days between consecutive composite dates):")
print(diffs.value_counts().sort_index())

naive_expected = (dates_sorted.max() - dates_sorted.min()).days // 8 + 1
true_grid = pd.date_range(f"{dates_sorted.min().year}-01-01", dates_sorted.max(), freq="8D")
true_grid = true_grid[(true_grid >= dates_sorted.min())]
print(f"\nNaive uniform-8-day-from-start expected dates: {naive_expected}")
print(f"Correct per-year MODIS 8-day grid expected dates: {len(true_grid)}")
print("(Always use the per-year grid -- the naive version drifts across year boundaries.)")


## 6. Per-pixel summary

In [ ]:
summary = layers["Lai_500m"].groupby("pixel").agg(
    n_rows=("date", "count"),
    n_fill=("is_fill", "sum"),
    first_date=("date", "min"),
    last_date=("date", "max"),
)
summary


## 7. Value ranges (valid values only)

In [ ]:
expected_ranges = {"Lai_500m": (0, 10), "Fpar_500m": (0, 1)}
for name, (lo, hi) in expected_ranges.items():
    df = layers[name]
    valid = df.loc[~df["is_fill"], "value"]
    n_out = ((valid < lo) | (valid > hi)).sum()
    print(f"{name}: min={valid.min():.3f}, max={valid.max():.3f}, mean={valid.mean():.3f} "
          f"-- {n_out} values outside plausible [{lo},{hi}] range")


## 8. Quality (FparLai_QC) breakdown

`FparLai_QC` is a packed byte. The SCF_QC "goodness" field lives in
**bits 5-7**: `(byte >> 5) & 0b111`. This is the one and only correct
way to read quality out of this byte -- treating the raw byte value
itself as a quality score (without this bit-shift) silently mixes in
unrelated bits (sensor, cloud state, dead-detector flag) and gives a
meaningless result. This same decode function is reused everywhere
else in this project that needs QC (see `02-correlation-analysis`).

In [ ]:
def decode_scf_qc(qc_byte):
    if pd.isna(qc_byte):
        return np.nan
    return (int(qc_byte) >> 5) & 0b111

goodness_labels = {0: "Best", 1: "Good", 2: "Fair", 3: "Poor", 4: "Failed"}

qc_df = layers["FparLai_QC"].copy()
qc_df["scf_qc"] = qc_df["raw"].apply(decode_scf_qc)

counts = qc_df["scf_qc"].value_counts().sort_index()
for level, n in counts.items():
    print(f"Level {int(level)} ({goodness_labels.get(int(level),'?')}): {n} observations ({n/len(qc_df):.1%})")

goodness_score = (4 - qc_df["scf_qc"].dropna()).mean() / 4
print(f"\nOverall goodness score: {goodness_score:.3f}  "
      f"(1.0 = all Best quality, 0.0 = all Failed)")


## 9. Summary

In [ ]:
print("=== MODIS FACT SHEET ===")
for name in ["Lai_500m", "Fpar_500m"]:
    df = layers[name]
    print(f"{name}: {df['date'].nunique()} dates x {df['pixel'].nunique()} pixels, "
          f"{100*df['is_fill'].mean():.1f}% fill")
print(f"QC goodness score: {goodness_score:.3f}")
